# Train Psychology Adapter — MediSign MedGemma 4B

**Chạy trên FPT Cloud Notebook (H100 80GB).**

Notebook này:
1. Kiểm tra GPU + cài dependencies (bao gồm `flash-attn`)
2. Login HuggingFace
3. Pull dataset OARS psychology từ HF
4. Pull source code từ GitHub (force re-clone — đồng nhất với Medical notebook)
5. Train Psychology Adapter với tối ưu H100: `bf16`, `tf32`, `flash_attention_2`, `gradient_checkpointing`, `dataloader_num_workers`, TensorBoard logging
6. Verify adapter output
7. Push adapter lên HF

**Trước khi chạy:** Sửa `HF_TOKEN` ở Cell Config.

**Cải thiện so với v1:**
- **Fix bug quan trọng:** Thêm `--model_id` (v1 thiếu → script dùng default model sai)
- **Fix re-clone:** Force re-clone thay vì conditional check (tránh dùng code cũ)
- `--bf16` + `--tf32`: tăng tốc 2–3× trên H100 Tensor Core
- `flash_attention_2`: giảm VRAM, tăng throughput
- `--gradient_checkpointing`: tiết kiệm VRAM
- `--dataloader_num_workers 8`: CPU prefetch song song, GPU không ngồi chờ
- TensorBoard logging mỗi 20 steps: theo dõi loss curve real-time
- Smoke test trước khi push

**Ước tính thời gian:** ~10–15 phút (giảm từ ~20 phút) trên H100 80GB với 1.7K records.

## 0. Kiểm tra GPU và môi trường

In [ ]:
import os, sys

print("=" * 60)
print("GPU INFO")
print("=" * 60)
!nvidia-smi

mem_gb = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / (1024**3)
cpu_cores = os.cpu_count()
print(f"\nSystem RAM : {mem_gb:.0f} GB")
print(f"CPU cores  : {cpu_cores}")
print(f"→ dataloader_num_workers sẽ dùng: {min(8, cpu_cores // 2)}")

## 1. Cài dependencies

> **Nếu đã chạy `train_medical_adapter.ipynb` trên cùng session:** Tất cả packages đã cài, cell này vẫn chạy được (pip sẽ skip những gì đã có).
>
> **Flash-Attention** cần compile từ source (~3–5 phút lần đầu).

In [ ]:
import subprocess, sys

def pip_install(args, label=""):
    print(f"Installing: {label or args[-1]} ...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q"] + args,
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"  ✅ OK")
    else:
        print(f"  ❌ FAILED:\n{result.stderr[-500:]}")
        raise RuntimeError(f"pip install failed: {args}")

pip_install(["--upgrade", "pip"], "pip")
pip_install(
    ["torch", "torchvision", "torchaudio",
     "--index-url", "https://download.pytorch.org/whl/cu124"],
    "torch + cu124"
)
pip_install(["transformers>=4.50", "peft>=0.13", "bitsandbytes>=0.44",
             "accelerate>=0.34", "trl>=0.12", "datasets>=3.0"], "core ML")
pip_install(["sentencepiece", "protobuf", "huggingface_hub"], "tokenizer libs")
pip_install(["tensorboard"], "tensorboard")
pip_install(["flash-attn", "--no-build-isolation"], "flash-attn (compile ~3-5 phút)")

print("\n✅ Tất cả dependencies đã cài xong")

In [ ]:
import torch

assert torch.cuda.is_available(), "❌ CUDA không available"

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU        : {gpu_name}")
print(f"VRAM       : {vram_gb:.1f} GB")
print(f"PyTorch    : {torch.__version__}")
print(f"CUDA       : {torch.version.cuda}")
print(f"BF16 support: {torch.cuda.is_bf16_supported()}")

try:
    import flash_attn
    print(f"Flash-Attn : {flash_attn.__version__} ✅")
except ImportError:
    print("Flash-Attn : ❌ — kiểm tra lại cell cài đặt")

## 2. Config tập trung

In [ ]:
import os, torch

# ============================================================
# TOKEN — CHỈNH Ở ĐÂY
# ============================================================
HF_TOKEN = "hf_YOUR_TOKEN_HERE"

# ============================================================
# MODEL — QUAN TRỌNG: phải chỉ định rõ, không dùng default
# ============================================================
BASE_MODEL_ID   = "google/medgemma-1.5-4b-it"
ADAPTER_REPO_ID = "thuaannn/medisign-medgemma4b-psychology"

# ============================================================
# DATA
# ============================================================
DATA_REPO_ID = "thuaannn/medisign-training-data"
DATA_DIR     = "data/training_clean/medgemma_4b"
TRAIN_FILE   = f"{DATA_DIR}/psychology_train.jsonl"
EVAL_FILE    = f"{DATA_DIR}/psychology_eval.jsonl"

# ============================================================
# OUTPUT
# ============================================================
CHECKPOINT_DIR = "output/medisign_medgemma4b_psychology/checkpoints"
ADAPTER_DIR    = "output/medisign_medgemma4b_psychology/adapter"
LOG_DIR        = "output/medisign_medgemma4b_psychology/logs"

# ============================================================
# TRAINING HYPERPARAMS
# Psychology dataset nhỏ hơn (1.7K records) → tune khác Medical
# ============================================================
NUM_EPOCHS    = 3
# Dataset nhỏ: batch lớn hơn để tránh noise gradient
BATCH_SIZE    = 4
GRAD_ACCUM    = 4      # effective batch = 16 (nhỏ hơn Medical vì dataset nhỏ)
LR            = 1e-4   # LR thấp hơn Medical để tránh overfit dataset nhỏ
MAX_SEQ_LEN   = 2048

# LoRA — rank cao hơn vì dataset nhỏ cần capacity model tốt
LORA_R        = 32
LORA_ALPHA    = 64
LORA_DROPOUT  = 0.1   # dropout cao hơn để regularize

# ============================================================
# H100 OPTIMIZATION FLAGS (giống Medical)
# ============================================================
USE_BF16                 = torch.cuda.is_bf16_supported()
USE_TF32                 = True
USE_FLASH_ATTN           = True
GRADIENT_CHECKPOINTING   = True
DATALOADER_NUM_WORKERS   = min(8, (os.cpu_count() or 4) // 2)
LOGGING_STEPS            = 10    # log dày hơn vì dataset nhỏ, ít steps hơn
SAVE_STEPS               = 50
EVAL_STEPS               = 50

# Bật TF32 ngay tại đây
torch.backends.cuda.matmul.allow_tf32 = USE_TF32
torch.backends.cudnn.allow_tf32       = USE_TF32

print("CONFIG SUMMARY")
print("=" * 50)
print(f"Base model            : {BASE_MODEL_ID}")
print(f"BF16                  : {USE_BF16}")
print(f"TF32                  : {USE_TF32}")
print(f"Flash-Attention 2     : {USE_FLASH_ATTN}")
print(f"Gradient checkpointing: {GRADIENT_CHECKPOINTING}")
print(f"Batch size (per GPU)  : {BATCH_SIZE}")
print(f"Gradient accumulation : {GRAD_ACCUM}")
print(f"Effective batch size  : {BATCH_SIZE * GRAD_ACCUM}")
print(f"Dataloader workers    : {DATALOADER_NUM_WORKERS}")
print(f"LoRA rank             : {LORA_R} (cao hơn Medical do dataset nhỏ)")
print(f"LR                    : {LR} (thấp hơn Medical)")
print(f"Epochs                : {NUM_EPOCHS}")
print(f"Max seq length        : {MAX_SEQ_LEN}")

## 3. Login HuggingFace

In [ ]:
os.environ["HF_TOKEN"]               = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

from huggingface_hub import login, whoami
login(token=HF_TOKEN, add_to_git_credential=False)

user_info = whoami(token=HF_TOKEN)
print(f"✅ Logged in as: {user_info['name']}")

## 4. Pull dataset Psychology từ HF

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path

Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

print(f"Pulling dataset từ {DATA_REPO_ID} ...")
snapshot_download(
    repo_id=DATA_REPO_ID,
    repo_type="dataset",
    local_dir=DATA_DIR,
    allow_patterns=["psychology_train.jsonl", "psychology_eval.jsonl"],
)

print("\nDataset summary:")
for fname in ["psychology_train.jsonl", "psychology_eval.jsonl"]:
    path = Path(DATA_DIR) / fname
    if not path.exists():
        raise FileNotFoundError(f"❌ Không tìm thấy: {path}")
    n = sum(1 for _ in path.open(encoding="utf-8"))
    size_mb = path.stat().st_size / 1024 / 1024
    print(f"  {fname}: {n:,} records ({size_mb:.1f} MB)")

# Kiểm tra thêm: 1 dòng đầu để verify format
import json
with open(TRAIN_FILE, encoding="utf-8") as f:
    sample = json.loads(f.readline())
print(f"\nSample record keys: {list(sample.keys())}")

## 5. Pull source code từ GitHub (force re-clone)

> **Fix so với v1:** Luôn force re-clone, không dùng conditional check.  
> Tránh trường hợp script cũ còn sót lại từ session trước.

In [ ]:
import shutil, subprocess

REPO_DIR     = "medisign_repo"
GITHUB_URL   = "https://github.com/VNDT1625/MediSign_AI.git"
TRAIN_SCRIPT = f"{REPO_DIR}/scripts/train_qlora_medgemma.py"

# ALWAYS force re-clone — đảm bảo code mới nhất từ main
if os.path.exists(REPO_DIR):
    print(f"Removing old repo at ./{REPO_DIR} ...")
    shutil.rmtree(REPO_DIR)

print(f"Cloning {GITHUB_URL} ...")
result = subprocess.run(
    ["git", "clone", "--depth", "1", GITHUB_URL, REPO_DIR],
    capture_output=True, text=True
)
if result.returncode != 0:
    raise RuntimeError(f"git clone failed:\n{result.stderr}")
print("✅ Clone thành công")

commit = subprocess.run(
    ["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
    capture_output=True, text=True
).stdout.strip()
print(f"Commit     : {commit}")

if not os.path.exists(TRAIN_SCRIPT):
    print("\n❌ Script không tìm thấy. Contents of scripts/:")
    for f in Path(f"{REPO_DIR}/scripts").iterdir():
        print(f"  {f.name}")
    raise FileNotFoundError(f"{TRAIN_SCRIPT} missing")

print(f"✅ Train script: {TRAIN_SCRIPT}")
sys.path.insert(0, os.path.abspath(f"{REPO_DIR}/scripts"))

## 6. Verify base model access

In [ ]:
from huggingface_hub import model_info

print(f"Checking access to {BASE_MODEL_ID} ...")
try:
    info = model_info(BASE_MODEL_ID, token=HF_TOKEN)
    print(f"✅ Accessible: {info.modelId}")
except Exception as e:
    print(f"❌ Không truy cập được: {e}")
    print("\n→ Vào https://huggingface.co/google/medgemma-1.5-4b-it")
    print("  Accept terms trước khi tiếp tục.")
    raise

## 7. Setup output directories & TensorBoard

In [ ]:
for d in [CHECKPOINT_DIR, ADAPTER_DIR, LOG_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

print("Output structure:")
print(f"  Checkpoints  : {CHECKPOINT_DIR}")
print(f"  Final adapter: {ADAPTER_DIR}")
print(f"  TensorBoard  : {LOG_DIR}")

%load_ext tensorboard
%tensorboard --logdir {LOG_DIR}

## 8. Train Psychology Adapter — H100 Optimized

**Fix quan trọng so với v1:** `--model_id` được chỉ định rõ ràng.  
V1 không có arg này → script có thể dùng default model khác `medgemma-1.5-4b-it`.

**Hyperparams tune cho dataset nhỏ (1.7K records):**
- LR thấp hơn (1e-4 vs 2e-4) → tránh overfit
- LoRA rank cao hơn (32 vs 16) → capacity tốt hơn cho domain nhỏ
- Dropout cao hơn (0.1 vs 0.05) → regularization
- Effective batch nhỏ hơn (16 vs 32) → gradient update dày hơn

In [ ]:
import time
start_time = time.time()

os.environ["TORCH_ALLOW_TF32"]        = "1"
os.environ["TOKENIZERS_PARALLELISM"]   = "false"

!python {REPO_DIR}/scripts/train_qlora_medgemma.py \
  --model_id            {BASE_MODEL_ID} \
  --train_file          {TRAIN_FILE} \
  --eval_file           {EVAL_FILE} \
  --num_epochs          {NUM_EPOCHS} \
  --per_device_train_batch_size {BATCH_SIZE} \
  --gradient_accumulation_steps {GRAD_ACCUM} \
  --learning_rate       {LR} \
  --max_seq_length      {MAX_SEQ_LEN} \
  --lora_r              {LORA_R} \
  --lora_alpha          {LORA_ALPHA} \
  --lora_dropout        {LORA_DROPOUT} \
  --bf16                {str(USE_BF16).lower()} \
  --tf32                {str(USE_TF32).lower()} \
  --attn_impl           flash_attention_2 \
  --gradient_checkpointing {str(GRADIENT_CHECKPOINTING).lower()} \
  --dataloader_num_workers {DATALOADER_NUM_WORKERS} \
  --group_by_length     true \
  --lr_scheduler_type   cosine \
  --warmup_ratio        0.05 \
  --save_steps          {SAVE_STEPS} \
  --eval_steps          {EVAL_STEPS} \
  --save_total_limit    3 \
  --load_best_model_at_end true \
  --report_to           tensorboard \
  --logging_dir         {LOG_DIR} \
  --logging_steps       {LOGGING_STEPS} \
  --output_dir          {CHECKPOINT_DIR} \
  --adapter_dir         {ADAPTER_DIR}

elapsed = time.time() - start_time
print(f"\n⏱ Training time: {elapsed/60:.1f} phút")

## 9. Verify adapter output

In [ ]:
from pathlib import Path
import json

adapter_dir = Path(ADAPTER_DIR)

if not adapter_dir.exists() or not any(adapter_dir.iterdir()):
    raise RuntimeError(f"❌ Adapter directory trống: {adapter_dir}")

print(f"Adapter files tại {adapter_dir}:")
total_mb = 0
for f in sorted(adapter_dir.iterdir()):
    size_mb = f.stat().st_size / 1024 / 1024
    total_mb += size_mb
    print(f"  {f.name:<40} {size_mb:>8.1f} MB")
print(f"  {'TOTAL':<40} {total_mb:>8.1f} MB")

config_path = adapter_dir / "adapter_config.json"
if config_path.exists():
    cfg = json.loads(config_path.read_text())
    print(f"\nAdapter config:")
    print(f"  peft_type  : {cfg.get('peft_type')}")
    print(f"  base_model : {cfg.get('base_model_name_or_path')}")
    print(f"  r          : {cfg.get('r')}")
    print(f"  lora_alpha : {cfg.get('lora_alpha')}")
    
    # Verify base model đúng — quan trọng vì v1 không truyền --model_id
    assert cfg.get('base_model_name_or_path') == BASE_MODEL_ID, (
        f"❌ base_model sai! Expected {BASE_MODEL_ID}, "
        f"got {cfg.get('base_model_name_or_path')}"
    )
    print(f"\n✅ base_model đúng: {BASE_MODEL_ID}")

## 10. Quick smoke test — load adapter và inference

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

print("Loading tokenizer ...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=HF_TOKEN)

print("Loading base model với flash_attention_2 + bf16 ...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    token=HF_TOKEN,
    torch_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
    attn_implementation="flash_attention_2" if USE_FLASH_ATTN else "eager",
    device_map="auto",
)

print("Merging adapter ...")
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()

# Test prompt liên quan đến tâm lý học / OARS
test_prompt = (
    "Tôi cảm thấy rất lo lắng và không thể ngủ được trong 2 tuần qua. "
    "Bạn có thể giúp tôi hiểu điều gì đang xảy ra không?"
)
inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)

with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.bfloat16):
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )

response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print(f"\nPrompt  : {test_prompt}")
print(f"Response: {response}")

del model, base_model
torch.cuda.empty_cache()
print("\n✅ Smoke test passed")

## 11. Push Psychology Adapter lên HuggingFace

In [ ]:
from huggingface_hub import HfApi, upload_folder

api = HfApi(token=HF_TOKEN)

try:
    api.create_repo(repo_id=ADAPTER_REPO_ID, exist_ok=True, private=False)
    print(f"Repo: https://huggingface.co/{ADAPTER_REPO_ID}")
except Exception as e:
    print(f"Note: {e}")

commit_msg = (
    f"v2: psychology OARS adapter, bf16+flash_attn2, "
    f"1.7K conversations, {NUM_EPOCHS} epochs, lora_r={LORA_R}"
)

print("\nUploading adapter ...")
upload_folder(
    folder_path=ADAPTER_DIR,
    repo_id=ADAPTER_REPO_ID,
    commit_message=commit_msg,
    token=HF_TOKEN,
)

print(f"\n✅ Pushed to https://huggingface.co/{ADAPTER_REPO_ID}")
print(f"Commit: {commit_msg}")

## Done!

Cả 2 adapter đã có trên HF:
- `thuaannn/medisign-medgemma4b-adapter` (Medical)
- `thuaannn/medisign-medgemma4b-psychology` (Psychology)

**Tối ưu đã áp dụng:**
- ✅ Fix bug: `--model_id` được chỉ định rõ
- ✅ Fix re-clone: force re-clone thay vì conditional
- ✅ BF16 precision + TF32 matmul
- ✅ Flash-Attention 2
- ✅ Gradient checkpointing
- ✅ Dataloader workers = 8
- ✅ Cosine LR scheduler + warmup 5% (cao hơn Medical cho dataset nhỏ)
- ✅ LoRA rank 32 (cao hơn Medical, phù hợp dataset nhỏ)
- ✅ Assert base_model đúng sau training
- ✅ TensorBoard logging mỗi 10 steps
- ✅ Smoke test trước khi push

Deploy trên server FPT Cloud:
```bash
MEDISIGN_ADAPTER_PATH=~/MediSign_AI/output/medisign_medgemma4b_medical/adapter \
MEDISIGN_PSYCHOLOGY_ADAPTER_PATH=~/MediSign_AI/output/medisign_medgemma4b_psychology/adapter \
bash scripts/cloud/start-fpt-medgemma.sh
```